In [ ]:
# !pip install protobuf==3.20.3 --user

In [ ]:
# !pip install mediapipe==0.10.14 --user

In [ ]:
# !pip install tensorflow==2.15.0 --user

In [ ]:
# !pip install jax==0.4.23 jaxlib==0.4.23 numpy==1.26.4 ml-dtypes==0.2.0 --user

In [ ]:
import cv2
import mediapipe as mp
import tensorflow as tf

print("Versi OpenCV:", cv2.__version__)
print("Versi MediaPipe:", mp.__version__)
print("Versi TensorFlow:", tf.__version__)


Versi OpenCV: 4.13.0
Versi MediaPipe: 0.10.14
Versi TensorFlow: 2.15.0


# Import Library dan Dependency

In [ ]:
!pip install opencv-python
!pip install mediapipe
!pip install tensorflow
!pip install numpy
!pip install matplotlib
!pip install scikit-learn

# Import Seluruh Library (CNN + LSTM)

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("Semua library berhasil di-import!")


Semua library berhasil di-import!


# Membaca Dataset Gambar Bahasa Isyarat

In [ ]:
dataset_path = "ai isyarat image"

print("Isi folder dataset:")
classes_abjad = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
print(classes_abjad)

print("\nJumlah gambar tiap kelas:")
total_images = 0

for label in classes_abjad:
    folder = os.path.join(dataset_path, label)
    images = os.listdir(folder)
    print(f"{label} : {len(images)} gambar")
    total_images += len(images)

print("\nTotal dataset gambar abjad:", total_images)

# Preprocessing, Normalisasi, dan Encoding Data Abjad

In [ ]:
img_size = 64
data = []
labels = []

for label in classes_abjad:
    folder = os.path.join(dataset_path, label)
    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (img_size, img_size))
        data.append(img)
        labels.append(label)

# Normalisasi piksel (0 - 1)
data = np.array(data, dtype="float32") / 255.0
labels = np.array(labels)

# Encoding label huruf A-Z menjadi angka 0-25
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

# Simpan daftar kelas abjad agar bisa dipanggil kapan saja oleh kamera
np.save("kelas_abjad.npy", encoder.classes_)

print("Total gambar diproses:", len(data))
print("Daftar kelas:", encoder.classes_)

# Pembagian Dataset Training & Testing + Augmentasi (Abjad)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    labels_encoded,
    test_size=0.2,
    random_state=42
)

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1
)
datagen.fit(X_train)

print("Data Training Abjad:", len(X_train))
print("Data Testing Abjad:", len(X_test))

# Pembuatan, Training, dan Penyimpanan Model CNN (Abjad)

In [ ]:
model = Sequential([
    Input(shape=(64, 64, 3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(256, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=20,
    validation_data=(X_test, y_test)
)

print("Training Accuracy :", history.history['accuracy'][-1])
print("Validation Accuracy :", history.history['val_accuracy'][-1])

model.save("model_isyarat.h5")
print("Model abjad berhasil disimpan sebagai 'model_isyarat.h5'!")

# Ekstraksi Fitur Gerakan dari Folder Dataset Kalimat

In [2]:
import cv2
import numpy as np
import os
import mediapipe as mp

VIDEO_PATH = "Dataset Kalimat"
DATA_KEYPOINTS_PATH = "Data_Keypoints"
SEQUENCE_LENGTH = 30 # Standar 30 frame

os.makedirs(DATA_KEYPOINTS_PATH, exist_ok=True)
mp_holistic = mp.solutions.holistic

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    kalimat_classes = [d for d in os.listdir(VIDEO_PATH) if os.path.isdir(os.path.join(VIDEO_PATH, d))]
    
    for action in kalimat_classes:
        action_path = os.path.join(VIDEO_PATH, action)
        os.makedirs(os.path.join(DATA_KEYPOINTS_PATH, action), exist_ok=True)
        
        for video_file in os.listdir(action_path):
            if video_file.endswith(('.mp4', '.avi', '.mov')):
                video_path = os.path.join(action_path, video_file)
                cap = cv2.VideoCapture(video_path)
                frames_data = []
                
                frame_count = 0
                while cap.isOpened() and frame_count < SEQUENCE_LENGTH:
                    ret, frame = cap.read()
                    if not ret: break
                    
                    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    image.flags.writeable = False                  
                    results = holistic.process(image)
                    
                    frames_data.append(extract_keypoints(results))
                    frame_count += 1
                    
                while len(frames_data) < SEQUENCE_LENGTH:
                    frames_data.append(frames_data[-1] if len(frames_data) > 0 else np.zeros(258))
                    
                np.save(os.path.join(DATA_KEYPOINTS_PATH, action, f"{video_file.split('.')[0]}.npy"), frames_data)
                cap.release()
        print(f"Berhasil ekstrak data kalimat: {action}")

Berhasil ekstrak data kalimat: .ipynb_checkpoints


C:\Users\NITRO\AppData\Roaming\Python\Python310\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Berhasil ekstrak data kalimat: Apa
Berhasil ekstrak data kalimat: Apa Kabar
Berhasil ekstrak data kalimat: Bagaimana
Berhasil ekstrak data kalimat: Baik
Berhasil ekstrak data kalimat: Belajar
Berhasil ekstrak data kalimat: Berapa
Berhasil ekstrak data kalimat: Berdiri
Berhasil ekstrak data kalimat: Bingung
Berhasil ekstrak data kalimat: Dia
Berhasil ekstrak data kalimat: Dimana
Berhasil ekstrak data kalimat: Duduk
Berhasil ekstrak data kalimat: Halo
Berhasil ekstrak data kalimat: Kalian
Berhasil ekstrak data kalimat: Kami
Berhasil ekstrak data kalimat: Kamu
Berhasil ekstrak data kalimat: Kapan
Berhasil ekstrak data kalimat: Kemana
Berhasil ekstrak data kalimat: Kita
Berhasil ekstrak data kalimat: Makan
Berhasil ekstrak data kalimat: Mandi
Berhasil ekstrak data kalimat: Marah
Berhasil ekstrak data kalimat: Melihat
Berhasil ekstrak data kalimat: Membaca
Berhasil ekstrak data kalimat: Menulis
Berhasil ekstrak data kalimat: Mereka
Berhasil ekstrak data kalimat: Minum
Berhasil ekstrak data 

# Preprocessing & Augmentasi Data Sekuensial Kalimat

In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# =================================================================
# BAGIAN 1: PILIH KOSAKATA YANG INGIN DIGUNAKAN
# =================================================================
DATA_PATH = os.path.join('Data_Keypoints')
SEQUENCE_LENGTH = 30 # Jumlah frame per video

# Daftar lengkap 39 kata yang tersedia di foldermu (sebagai contekan):
# 'Apa', 'Apa Kabar', 'Bagaimana', 'Baik', 'Belajar', 'Berapa', 'Berdiri', 
# 'Bingung', 'Dia', 'Dimana', 'Duduk', 'Halo', 'Kalian', 'Kami', 'Kamu', 
# 'Kapan', 'Kemana', 'Kita', 'Makan', 'Mandi', 'Marah', 'Melihat', 'Membaca', 
# 'Menulis', 'Mereka', 'Minum', 'Pendek', 'Ramah', 'Sabar', 'Saya', 'Sedih', 
# 'Selamat Malam', 'Selamat Pagi', 'Selamat Siang', 'Selamat Sore', 'Senang', 
# 'Siapa', 'Tidur', 'Tinggi'

# Masukkan HANYA kata-kata yang ingin kamu pakai di dalam kurung siku ini:
actions = np.array([
    'Halo', 
    'Apa Kabar', 
    'Baik', 
    'Saya', 
    'Kamu', 
    'Belajar', 
    'Makan', 
    'Minum', 
    'Selamat Pagi', 
    'Senang'
])

print(f"Total kelas yang dipilih: {len(actions)} kata/kalimat.")
print(f"Daftar kelas aktif: {actions}\n")

sequences, labels_kalimat = [], []
for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    if not os.path.exists(action_path):
        print(f"PERINGATAN: Folder '{action}' tidak ditemukan! Cek ejaan huruf besar/kecilnya.")
        continue
        
    # Mengambil langsung semua file berekstensi .npy dari folder yang dipilih
    for file_name in os.listdir(action_path):
        if file_name.endswith('.npy'):
            res = np.load(os.path.join(action_path, file_name))
            sequences.append(res)
            labels_kalimat.append(action)

# =================================================================
# BAGIAN 2: AUGMENTASI DAN PENYIAPAN LABEL
# =================================================================
unique_kalimat = sorted(list(set(labels_kalimat)))

# Tambahkan kelas 'Netral' (kondisi diam)
if "Netral" not in unique_kalimat:
    unique_kalimat = ["Netral"] + unique_kalimat
    for _ in range(10):
        idle_seq = np.random.uniform(0.0, 0.02, size=(SEQUENCE_LENGTH, 258)).astype("float32")
        sequences.append(idle_seq)
        labels_kalimat.append("Netral")

label_map = {label: num for num, label in enumerate(unique_kalimat)}
np.save("kelas_kalimat.npy", np.array(unique_kalimat))

X_seq = []
y_seq = []

# Augmentasi otomatis: perbanyak variasi data video 
for seq, lbl in zip(sequences, labels_kalimat):
    seq_arr = np.array(seq, dtype="float32")
    X_seq.append(seq_arr)
    y_seq.append(label_map[lbl])
    
    for _ in range(15):
        noise = np.random.normal(0, 0.015, seq_arr.shape).astype("float32")
        scale = np.random.uniform(0.95, 1.05)
        X_seq.append(np.clip(seq_arr * scale + noise, 0.0, 1.0))
        y_seq.append(label_map[lbl])

X_kalimat = np.array(X_seq, dtype="float32")
y_kalimat = to_categorical(y_seq, num_classes=len(unique_kalimat))

X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_kalimat, y_kalimat, test_size=0.2, random_state=42, stratify=y_seq
)

print("="*50)
print("Daftar Kelas Kalimat (Termasuk Netral):", unique_kalimat)
print("Total Sampel Sequence (setelah augmentasi):", len(X_kalimat))
print("Shape X_train Kalimat:", X_train_k.shape)
print("Shape X_test Kalimat:", X_test_k.shape)


Total kelas yang dipilih: 10 kata/kalimat.
Daftar kelas aktif: ['Halo' 'Apa Kabar' 'Baik' 'Saya' 'Kamu' 'Belajar' 'Makan' 'Minum'
 'Selamat Pagi' 'Senang']

Daftar Kelas Kalimat (Termasuk Netral): ['Netral', 'Apa Kabar', 'Baik', 'Belajar', 'Halo', 'Kamu', 'Makan', 'Minum', 'Saya', 'Selamat Pagi', 'Senang']
Total Sampel Sequence (setelah augmentasi): 8160
Shape X_train Kalimat: (6528, 30, 258)
Shape X_test Kalimat: (1632, 30, 258)


# Pembuatan, Training & Penyimpanan Model LSTM Kalimat

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

# 1. Membangun Arsitektur Otak AI (LSTM)
model_kalimat = Sequential([
    Input(shape=(SEQUENCE_LENGTH, 258)),
    LSTM(64, return_sequences=True, activation='tanh'),
    Dropout(0.2),
    LSTM(128, return_sequences=False, activation='tanh'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(len(unique_kalimat), activation='softmax')
])

# 2. Menentukan metode evaluasi belajar AI
model_kalimat.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Menampilkan struktur model
model_kalimat.summary()

print("\nMemulai proses pelatihan AI (Training) selama 60 putaran...")
print("Silakan tunggu, proses ini akan memakan waktu beberapa menit.\n")

# 3. Memulai proses belajar (Training)
history_kalimat = model_kalimat.fit(
    X_train_k, y_train_k,
    epochs=60,
    batch_size=16,
    validation_data=(X_test_k, y_test_k)
)

print("\n=======================================================")
print("Training Accuracy Kalimat :", history_kalimat.history['accuracy'][-1])
print("Validation Accuracy Kalimat :", history_kalimat.history['val_accuracy'][-1])

# 4. Menyimpan model agar bisa digunakan di kamera nanti
model_kalimat.save("model_kalimat.h5")
print("=======================================================")
print("Mantap! Model kalimat berhasil disimpan sebagai 'model_kalimat.h5'!")



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 30, 64)            82688     
                                                                 
 dropout (Dropout)           (None, 30, 64)            0         
                                                                 
 lstm_1 (LSTM)               (None, 128)               98816     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 11)               

C:\Users\NITRO\AppData\Roaming\Python\Python310\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


# Sistem Deteksi, Penerjemah & Respon AI Real-Time

In [ ]:
import cv2
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.models import load_model

# --- FIX ERROR MEDIAPIPE ---
import mediapipe.python.solutions.holistic as mp_holistic
import mediapipe.python.solutions.drawing_utils as mp_drawing

# 1. Load Kedua Model
model_abjad = load_model("model_isyarat.h5", compile=False)
model_kalimat = load_model("model_kalimat.h5", compile=False)

kelas_abjad = np.load("kelas_abjad.npy", allow_pickle=True) if os.path.exists("kelas_abjad.npy") else sorted(os.listdir("ai isyarat image"))
kelas_kalimat = np.load("kelas_kalimat.npy", allow_pickle=True)

# 2. Fungsi Ekstraksi Keypoints
def get_cam_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

# Fungsi cek apakah ada tangan di dalam kotak hijau (Mode Abjad)
def cek_tangan_di_kotak(results, w, h, x1, y1, x2, y2):
    for hand_landmarks in [results.left_hand_landmarks, results.right_hand_landmarks]:
        if hand_landmarks is not None:
            # Ambil titik tengah telapak tangan (landmark nomor 9)
            cx = int(hand_landmarks.landmark[9].x * w)
            cy = int(hand_landmarks.landmark[9].y * h)
            if x1 <= cx <= x2 and y1 <= cy <= y2:
                return True
    return False

mode_aktif = "KALIMAT" 
flip_input_ai = False # Tekan 'F' untuk membalik orientasi tangan
gunakan_rgb_abjad = False # Tekan 'W' untuk ganti warna BGR/RGB
sequence_buffer = []
kalimat_hasil = "" 
prediksi_sementara = "-"
conf_sementara = 0.0

huruf_stabil = ""
hitung_frame = 0
frame_counter = 0
frames_tanpa_tangan = 0

cap = cv2.VideoCapture(0)
holistic_cam = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

while cap.isOpened():
    ret, raw_frame = cap.read()
    if not ret: break
    frame_counter += 1
    h, w, _ = raw_frame.shape

    # --- MODE 1: KALIMAT & KATA UTUH (LSTM) ---
    if mode_aktif == "KALIMAT":
        frame = cv2.flip(raw_frame, 1) if flip_input_ai else raw_frame.copy()
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = holistic_cam.process(image)
        image.flags.writeable = True

        mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
        mp_drawing.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        ada_tangan = (results.left_hand_landmarks is not None) or (results.right_hand_landmarks is not None)

        if ada_tangan:
            frames_tanpa_tangan = 0
            feat = get_cam_keypoints(results)
            sequence_buffer.append(feat)
            sequence_buffer = sequence_buffer[-30:]

            if len(sequence_buffer) == 30 and frame_counter % 3 == 0:
                input_tensor = tf.constant(np.expand_dims(sequence_buffer, axis=0), dtype=tf.float32)
                pred = model_kalimat(input_tensor, training=False).numpy()[0]
                idx = np.argmax(pred)
                conf = pred[idx]
                label = str(kelas_kalimat[idx])
                
                prediksi_sementara = label
                conf_sementara = conf

                if conf > 0.70 and label != "Netral":
                    kalimat_hasil = label
        else:
            frames_tanpa_tangan += 1
            # Jika tangan sudah diturunkan selama > 8 frame, hapus terjemahan otomatis
            if frames_tanpa_tangan > 8:
                sequence_buffer = []
                prediksi_sementara = "-"
                conf_sementara = 0.0
                kalimat_hasil = ""

        display_frame = frame if flip_input_ai else cv2.flip(frame, 1)

        status_flip = "CERMIN" if flip_input_ai else "NORMAL (Asli)"
        cv2.putText(display_frame, f"MODE KALIMAT | Orientasi: {status_flip}", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        
        teks_live = f"Deteksi Live: {prediksi_sementara} ({conf_sementara*100:.0f}%)" if prediksi_sementara != "-" else "Deteksi Live: Menunggu gerakan tangan..."
        cv2.putText(display_frame, teks_live, (20, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)

    # --- MODE 2: EJAAN ABJAD PER HURUF (CNN) ---
    elif mode_aktif == "ABJAD":
        display_frame = cv2.flip(raw_frame, 1)
        x1, y1, x2, y2 = w - 290, 90, w - 30, 350
        
        # Gunakan MediaPipe untuk mengecek apakah tangan ada di dalam kotak hijau
        image_abjad = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)
        image_abjad.flags.writeable = False
        results_abjad = holistic_cam.process(image_abjad)
        
        tangan_di_kotak = cek_tangan_di_kotak(results_abjad, w, h, x1, y1, x2, y2)
        
        roi = display_frame[y1:y2, x1:x2].copy()
        if flip_input_ai:
            roi = cv2.flip(roi, 1)

        if tangan_di_kotak:
            frames_tanpa_tangan = 0
            warna_kotak = (0, 255, 0) # Hijau saat tangan terdeteksi di dalam kotak
            
            roi_proses = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB) if gunakan_rgb_abjad else roi
            roi_input = cv2.resize(roi_proses, (64, 64)).astype("float32") / 255.0
            roi_input = np.reshape(roi_input, (1, 64, 64, 3))

            input_tensor_a = tf.constant(roi_input, dtype=tf.float32)
            pred_a = model_abjad(input_tensor_a, training=False).numpy()[0]
            idx_a = np.argmax(pred_a)
            conf_a = pred_a[idx_a]
            tebakan_huruf = str(kelas_abjad[idx_a])

            if conf_a > 0.50:
                if tebakan_huruf == huruf_stabil:
                    hitung_frame += 1
                else:
                    huruf_stabil = tebakan_huruf
                    hitung_frame = 0

                # Tampilkan HANYA 1 huruf saat ini (bukan += yang menyambung panjang)
                if hitung_frame >= 6:
                    kalimat_hasil = tebakan_huruf
                    
            teks_pred_abjad = f"Prediksi: {tebakan_huruf} ({conf_a*100:.0f}%)"
        else:
            frames_tanpa_tangan += 1
            warna_kotak = (0, 255, 255) # Kuning saat menunggu tangan masuk kotak
            teks_pred_abjad = "Masukkan tangan ke kotak"
            hitung_frame = 0
            huruf_stabil = ""
            
            # Jika tangan keluar dari kotak, langsung hilangkan tulisan Terjemahan
            if frames_tanpa_tangan > 5:
                kalimat_hasil = ""

        cv2.rectangle(display_frame, (x1, y1), (x2, y2), warna_kotak, 2)

        # Preview kecil di pojok kiri atas
        preview_kecil = cv2.resize(roi, (80, 80))
        display_frame[85:165, 20:100] = preview_kecil
        cv2.rectangle(display_frame, (20, 85), (100, 165), (255, 255, 0), 1)

        status_flip = "BALIK" if flip_input_ai else "CERMIN ASLI"
        status_warna = "RGB" if gunakan_rgb_abjad else "BGR"
        
        cv2.putText(display_frame, teks_pred_abjad, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, warna_kotak, 2)
        cv2.putText(display_frame, f"MODE ABJAD | Orientasi [F]: {status_flip} | Warna [W]: {status_warna}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
        cv2.putText(display_frame, "Isi penuh kotak dengan tanganmu", (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

    # --- TAMPILAN PANEL BAWAH ---
    cv2.rectangle(display_frame, (0, h - 80), (w, h), (20, 20, 20), -1)
    
    # Hanya muncul saat tangan sedang memperagakan huruf/kata, dan hilang saat selesai
    if kalimat_hasil != "":
        cv2.putText(display_frame, f"Terjemahan : {kalimat_hasil}", (15, h - 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    cv2.putText(display_frame, "[M] Mode | [F] Balik Tangan | [W] Warna Abjad | [C] Reset | [ESC] Keluar", (15, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

    cv2.imshow("Sign Bridge AI - Lomba", display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27: # ESC
        break
    elif key in [ord('m'), ord('M')]:
        mode_aktif = "ABJAD" if mode_aktif == "KALIMAT" else "KALIMAT"
        sequence_buffer = []
        kalimat_hasil = ""
        prediksi_sementara = "-"
    elif key in [ord('f'), ord('F')]:
        flip_input_ai = not flip_input_ai
        sequence_buffer = []
    elif key in [ord('w'), ord('W')]:
        gunakan_rgb_abjad = not gunakan_rgb_abjad
    elif key in [ord('c'), ord('C'), 13]:
        kalimat_hasil = ""
        sequence_buffer = []
        prediksi_sementara = "-"

holistic_cam.close()
cap.release()
cv2.destroyAllWindows()